# Lumis: YAML-led investigation and operational graph

This is an offline SDK walkthrough, not an embedded application cookbook or live GridCast qualification. It makes no network or paid model calls. It creates synthetic inputs in a temporary directory and checks supported **but unconfirmed** explanations.

From the SDK checkout: `uv sync --locked --all-groups`. Use that environment as your notebook kernel (Python 3.11+); install your preferred notebook UI separately if needed. The package currently on PyPI may predate this development architecture.

Run cells in order. Jupyter supports top-level `await`; do not call `asyncio.run` inside its existing loop. The tests execute these same cells without requiring Jupyter.

In [ ]:
import json
import tempfile
from pathlib import Path

import networkx as nx
import yaml

from lumis_sdk.core import Incident
from lumis_sdk.runtime import YamlProject
from lumis_sdk.runtime.scaffold import starter_documents

workspace = Path(tempfile.mkdtemp(prefix="lumis-notebook-"))
print("Temporary, synthetic-only workspace:", workspace)

## 1. Declare inputs once in YAML

The starter has one service, a registered health query and a falsifiable rule candidate. We explicitly opt into its synthetic observations file; the unmodified starter abstains without observations.

No source adapter is enabled here. Replace `sources` with approved Kubernetes/Prometheus/OTLP settings only when you want external access. Enabled sources run at preparation; merely declaring a model does not invoke it.

In [ ]:
for name, content in starter_documents().items():
    (workspace / name).write_text(content, encoding="utf-8")
config = json.loads((workspace / "lumis.yaml").read_text(encoding="utf-8"))
config["observations_file"] = "observations.json"
(workspace / "lumis.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
incident = Incident.model_validate_json((workspace / "incident.json").read_text(encoding="utf-8"))
print((workspace / "lumis.yaml").read_text(encoding="utf-8"))

## 2. Use the two-line API

Sources, candidate definitions, query catalog and optional replay input come from YAML. This automatically discovers/binds topology before investigation. `use_model=True` would be explicit paid-source opt-in; we leave it off.

In [ ]:
project = YamlProject.from_file(workspace / "lumis.yaml")
result = await project.investigate(incident)
assert result.outcome == "supported"
assert result.truth_state == "unconfirmed_hypothesis"
print(result.model_dump_json(indent=2))

## 3. Inspect source readiness and NetworkX graph

Retain a prepared snapshot when you want repeated investigation without rediscovery. `to_networkx()` returns an independent deep copy; it cannot mutate the kernel graph. DOT text can be rendered using your own graph viewer, without adding plotting dependencies to the SDK.

In [ ]:
prepared = await project.prepare(at=incident.ended_at)
assert prepared.discovery.complete
print(prepared.discovery.model_dump_json(indent=2))
neighborhood = prepared.graph.dependencies_within("service:demo", hops=2)
print(neighborhood.model_dump_json(indent=2))
nx_graph = prepared.graph.to_networkx()
assert isinstance(nx_graph, nx.MultiDiGraph)
print("Nodes:", list(nx_graph.nodes(data=True)))
print("Edges:", list(nx_graph.edges(keys=True, data=True)))
print(prepared.graph.to_dot())
missing = await prepared.investigate(incident, observations=())
assert missing.outcome == "abstained"
print("Withheld observations:", missing.outcome)

## 4. A second, independent estate: dataset lineage

Dependency/data flow is input dataset → preparation job → output dataset. This tests portable graph scoping, not automatic OpenLineage ingestion. The graph can hold jobs/models/services/resources; provenance describes what was actually declared or observed.

In [ ]:
lineage_config = {
    "project": {"name": "data-estate"},
    "graph": {
        "entities": [
            {"id": "dataset:raw", "kind": "dataset", "name": "Raw inputs"},
            {"id": "job:features", "kind": "job", "name": "Feature preparation"},
            {"id": "dataset:features", "kind": "dataset", "name": "Feature dataset"},
        ],
        "relationships": [
            {
                "source": "dataset:raw",
                "target": "job:features",
                "kind": "feeds",
                "provenance": ["operator-declaration"],
            },
            {
                "source": "job:features",
                "target": "dataset:features",
                "kind": "produces",
                "provenance": ["operator-declaration"],
            },
        ],
    },
}
lineage_path = workspace / "lineage.yaml"
lineage_path.write_text(yaml.safe_dump(lineage_config), encoding="utf-8")
data_estate = await YamlProject.from_file(lineage_path).prepare()
assert data_estate.graph.upstream_of("dataset:features") == ("dataset:raw", "job:features")
assert data_estate.graph.downstream_of("dataset:raw") == ("dataset:features", "job:features")
scoped = data_estate.graph.dependencies_within("dataset:features", hops=1)
assert {entity.id for entity in scoped.entities} == {"dataset:features", "job:features"}
print("Topological order:", list(nx.topological_sort(data_estate.graph.to_networkx())))
print(data_estate.graph.to_dot())

## 5. Move to an external cookbook

Read [Python API](../python-api.md), [YAML fields](../configuration.md), [graph semantics](../graph.md) and the [integration readiness checklist](../integrations.md).

A first read-only cookbook slice can use Kubernetes resources and existing Prometheus service-graph/evidence metrics. Scope the endpoint, namespace and operator-owned queries correctly. Live telemetry/access/model quality must be qualified separately. Git/recent-change, Loki/Tempo query adapters, Prefect and automatic OpenLineage ingestion remain roadmap work; a draft recipe listing them is not evidence of current support.

Inspect the temporary files if useful, then remove only this generated workspace when finished. No original user files were modified.